# Padel Stroke Classification from Skeletons (RTMO + ST-GCN)

This notebook turns the pose estimation insights from `pose-rtmlib.ipynb` into a full skeleton-based classifier: it extracts RTMO keypoints for each stroke clip, packs them into ST-GCN tensors, fine-tunes a pre-trained action recognition backbone, and exports evaluation-ready metrics.

## Roadmap
1. **Environment setup** – install PyTorch, MMEngine, and MMAction2 (provides pre-trained ST-GCN weights).
2. **Dataset discovery** – list padel shot categories available in `dataset/`.
3. **Pose extraction** – batch export RTMO-s keypoints for every video (skip if you already cached JSON/NPZ files).
4. **Build ST-GCN tensors** – convert raw keypoints into `(N, C, T, V, M)` format with skeletal connectivity metadata.
5. **Fine-tune pre-trained ST-GCN** – start from the NTU-RGBD checkpoint and adapt to padel strokes with class-balanced sampling.
6. **Evaluate & export** – confusion matrix, per-class metrics, and ONNX/TorchScript export for deployment.
7. **Player-of-interest heuristic** – automatically pick the striker in frames with up to four players.

In [ ]:
# Optional: run once per environment (skip if deps already installed)
import sys
if 'google.colab' in sys.modules:
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    !pip install -q mmengine mmcv-lite mmaction2 rtmlib decord rich tqdm
else:
    print('✓ Assume local environment already has PyTorch, MMEngine, MMAction2, and rtmlib installed.')

In [ ]:
from __future__ import annotations

import json
import math
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import pkg_resources

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from mmengine import Config
from mmengine.registry import init_default_scope
from mmaction.apis import init_recognizer, train_model, inference_recognizer

from rtmlib import RTMO

DATASET_ROOT = Path('../dataset').resolve()
POSE_CACHE = Path('../artifacts/poses')
SKELETON_CACHE = Path('../artifacts/stgcn')
POSE_CACHE.mkdir(parents=True, exist_ok=True)
SKELETON_CACHE.mkdir(parents=True, exist_ok=True)

## 1. Inspect available stroke categories
Ensure the directory structure matches `dataset/<stroke_name>/<clip>.mp4`.

In [ ]:
stroke_dirs = sorted([p for p in DATASET_ROOT.iterdir() if p.is_dir()])
stroke_inventory = {p.name: len(list(p.glob('*.mp4'))) for p in stroke_dirs}
pd.DataFrame.from_dict(stroke_inventory, orient='index', columns=['clips']).sort_values('clips', ascending=False)

## 2. Batch RTMO-s keypoint extraction
We reuse the RTMO-s (Moderate thresholds) setup from the benchmarking notebook and dump keypoints per clip. Each output JSON stores: video metadata, per-frame keypoints shaped `(people, joints, 3)` and confidence scores.

In [ ]:
def export_keypoints(video_path: Path, overwrite: bool = False) -> Path:
    target_path = POSE_CACHE / f"{video_path.stem}.json"

    if target_path.exists() and not overwrite:
        return target_path

    rtmo = RTMO(
        onnx_model='https://download.openmmlab.com/mmpose/v1/projects/rtmo/onnx_sdk/rtmo-s_8xb32-600e_body7-640x640-dac2bf74_20231211.zip',
        backend='onnxruntime',
        device='cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu',
        model_input_size=(640, 640),
        score_thr=0.2,
        nms_thr=0.45
    )

    import cv2
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        keypoints, scores = rtmo(frame)
        frames.append({
            'keypoints': np.asarray(keypoints).tolist() if keypoints is not None else [],
            'scores': np.asarray(scores).tolist() if scores is not None else []
        })
    cap.release()
    payload = {
        "video": video_path.name,
        "fps": fps,
        "frames": frames,
    }
    target_path.write_text(json.dumps(payload))
    return target_path

In [ ]:
def batch_export(overwrite: bool = False):
    manifest = []
    for stroke_dir in stroke_dirs:
        for video_path in sorted(stroke_dir.glob('*.mp4')):
            pose_path = export_keypoints(video_path, overwrite)
            manifest.append({'stroke': stroke_dir.name, 'video': video_path.name, 'pose_path': str(pose_path)})
    df = pd.DataFrame(manifest)
    df.to_csv(POSE_CACHE / 'manifest.csv', index=False)
    return df

pose_manifest = batch_export(overwrite=False)
pose_manifest.head()

## 3. Convert to ST-GCN tensors
ST-GCN expects tensors shaped `(C, T, V, M)` per sample, where `C=3` (x, y, score), `T` is number of frames, `V=17` joints, and `M` is the number of persons kept (we limit to the striker).

In [ ]:
COCO_LINKS = [(0,1),(1,2),(2,3),(3,4),(1,5),(5,6),(6,7),(1,8),(8,9),(9,10),(10,11),(8,12),(12,13),(13,14),(0,15),(0,16)]
MAX_PERSONS = 1  # keep highest-confidence track (player-of-interest)

def select_player(frame_entry):
    if not frame_entry['keypoints']:
        return np.zeros((MAX_PERSONS, 17, 3), dtype=np.float32)
    people = np.asarray(frame_entry['scores']).mean(axis=2)
    idx = np.argmax(people)
    kp = np.asarray(frame_entry['keypoints'])[idx:idx+1]
    return kp.astype(np.float32)

def build_sample(pose_json: Path) -> np.ndarray:
    payload = json.loads(pose_json.read_text())
    frames = payload['frames']
    tensors = []
    for frame in frames:
        kp = select_player(frame)  # shape (1, V, 3)
        tensors.append(kp)
    if not tensors:
        return np.zeros((3, 1, 17, MAX_PERSONS), dtype=np.float32)
    arr = np.stack(tensors, axis=1)  # (1, T, V, 3)
    arr = arr.transpose(3,1,2,0)  # -> (C, T, V, M)
    return arr

def cache_stgcn_tensors(manifest: pd.DataFrame):
    data = []
    for row in manifest.itertuples():
        pose_path = Path(row.pose_path)
        tensor = build_sample(pose_path)
        out_path = SKELETON_CACHE / f"{pose_path.stem}.npy"

        np.save(out_path, tensor)
        data.append({'stroke': row.stroke, 'tensor_path': str(out_path)})
    df = pd.DataFrame(data)
    df.to_csv(SKELETON_CACHE / 'labels.csv', index=False)
    return df

tensor_manifest = cache_stgcn_tensors(pose_manifest)
tensor_manifest.head()

## 4. Dataset wrapper & dataloaders
A thin PyTorch `Dataset` loads cached tensors, applies temporal cropping/padding, and encodes labels.

In [ ]:
class PadelSkeletonDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame, clip_len: int = 64):
        self.manifest = manifest.reset_index(drop=True)
        self.clip_len = clip_len
        self.labels = sorted(manifest['stroke'].unique())
        self.label2id = {label: idx for idx, label in enumerate(self.labels)}

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]
        tensor = np.load(row['tensor_path'])  # (C, T, V, M)
        tensor = self._crop_or_pad(tensor)
        return {
            'keypoint': torch.from_numpy(tensor).float(),
            'label': torch.tensor(self.label2id[row['stroke']], dtype=torch.long)
        }

    def _crop_or_pad(self, tensor):
        C, T, V, M = tensor.shape
        if T == self.clip_len:
            return tensor
        if T > self.clip_len:
            start = np.random.randint(0, T - self.clip_len + 1)
            return tensor[:, start:start+self.clip_len]
        pad = np.zeros((C, self.clip_len, V, M), dtype=tensor.dtype)
        pad[:, :T] = tensor
        return pad

train_df = tensor_manifest.sample(frac=0.8, random_state=42)
val_df = tensor_manifest.drop(train_df.index)
train_ds = PadelSkeletonDataset(train_df)
val_ds = PadelSkeletonDataset(val_df)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)

## 5. Initialize pre-trained ST-GCN backbone
MMAction2 ships with ST-GCN checkpoints trained on NTU-RGBD and Kinetics-Skeleton. We start from `stgcn_80e_ntu60_xsub_keypoint` and fine-tune the classification head.

In [ ]:
try:
    mmaction_pkg = Path(pkg_resources.resource_filename('mmaction', ''))
except Exception:
    mmaction_pkg = Path('../mmaction2').resolve()  # fallback to local clone if pip pkg not found
config_path = mmaction_pkg / 'configs' / 'skeleton' / 'stgcn' / 'stgcn_80e_ntu60_xsub_keypoint.py'
assert config_path.exists(), f'Config not found at {config_path}. Install mmaction2 or clone the repo.'
checkpoint_url = 'https://download.openmmlab.com/mmaction/skeleton/stgcn/stgcn_80e_ntu60_xsub_keypoint-9ba9be7c.pth'
cfg = Config.fromfile(str(config_path))
cfg.model.cls_head.num_classes = len(train_ds.labels)
cfg.train_dataloader.dataset.ann_file = str(SKELETON_CACHE / 'train.pkl')
cfg.val_dataloader.dataset.ann_file = str(SKELETON_CACHE / 'val.pkl')
cfg.test_dataloader.dataset.ann_file = str(SKELETON_CACHE / 'val.pkl')
cfg.work_dir = str(SKELETON_CACHE / 'work_dirs' / 'stgcn_padel')
cfg.schedule_cfg.max_epochs = 40
cfg.train_cfg.val_interval = 1
cfg.optim_wrapper.optimizer.lr = 0.01 * (len(train_ds) / 400)
init_default_scope('mmaction')
model = init_recognizer(cfg, checkpoint=checkpoint_url, device='cuda' if torch.cuda.is_available() else 'cpu')
print(model.cls_head.fc_cls.out_features, 'classes ready')

## 6. Fine-tune with stratified sampling
We feed the PyTorch loaders through MMAction2's custom dataset hooks or write a light training loop. Below is a Lightning-style loop that freezes the first two GCN stages to cope with the limited dataset size.

In [ ]:
def freeze_backbone_layers(model, stages_to_freeze=2):
    frozen = 0
    for name, param in model.backbone.named_parameters():
        if name.startswith('gcn_layers') and frozen < stages_to_freeze:
            param.requires_grad_(False)
        elif 'gcn_layers' not in name:
            continue
        else:
            break
freeze_backbone_layers(model)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=0.01)
device = next(model.parameters()).device

for epoch in range(20):
    model.train()
    for batch in train_loader:
        keypoint = batch['keypoint'].to(device)
        label = batch['label'].to(device)
        preds = model(keypoint=keypoint, label=label, mode='loss')['loss_cls']
        optimizer.zero_grad()
        preds.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        losses, correct, total = [], 0, 0
        for batch in val_loader:
            out = model(keypoint=batch['keypoint'].to(device), label=batch['label'].to(device), mode='loss')
            loss = out['loss_cls'].item()
            scores = model(keypoint=batch['keypoint'].to(device), mode='predict')['pred_scores']
            preds = scores.argmax(dim=1).cpu()
            correct += (preds == batch['label']).sum().item()
            total += len(preds)
            losses.append(loss)
    print(f"Epoch {epoch+1}: val_loss={np.mean(losses):.3f}, val_acc={correct/total:.2%}")

## 7. Evaluation & confusion matrix
Gather per-class precision/recall and export the tuned checkpoint.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(model, dataset, loader):
    device = next(model.parameters()).device
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            scores = model(keypoint=batch['keypoint'].to(device), mode='predict')['pred_scores']
            preds = scores.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(batch['label'].numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    print(classification_report(y_true, y_pred, target_names=dataset.labels))
    cm = confusion_matrix(y_true, y_pred)
    return cm

cm = evaluate(model, val_ds, val_loader)
cm

## 8. Export & deployment
Once satisfied, export to TorchScript/ONNX and log metrics for comparison with raw-video methods.

In [ ]:
export_path = SKELETON_CACHE / 'stgcn_padel_best.pth'
torch.save(model.state_dict(), export_path)
print('Saved to', export_path)

example = next(iter(val_loader))['keypoint'][:1].to(device)
torch.onnx.export(model, example, SKELETON_CACHE / 'stgcn_padel.onnx',
                  input_names=['keypoint'], output_names=['logits'],
                  opset_version=17)
print('ONNX model exported')

## 9. Player-of-interest detection (multi-player clips)
When four players appear, automatically select the striker by combining: (1) highest skeleton confidence near the ball, (2) proximity to the ball trajectory, and (3) hand velocity spikes. Optionally, run a lightweight tracker (ByteTrack or SORT) on person boxes, then associate each skeleton to the tracked IDs and mark the hitter as the player whose wrist speed crosses a threshold within ±12 frames of the annotated stroke timestamp.

## 10. Tips for small datasets
- Start from pre-trained weights (`stgcn_80e_ntu60_xsub`) and freeze early stages.
- Use mixup / CutMix across skeleton tensors to augment poses.
- Apply temporal jittering (random start, speed perturbation) to fight overfitting.
- Embrace k-fold cross-validation because each stroke type might only have a handful of clips.
- Track experiment metadata (thresholds, aggregation rules) in `artifacts/experiments.json` for reproducibility.